Evaluate urgency classification, undertriage and overtriage against case reference labels.


In [1]:
import os, sys
import numpy as np
import pandas as pd
from scipy import stats

def _build_data_tools():
    import json
    import os
    import re
    import unicodedata
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    GOLD = str(_repo)
    PAPER = str(_repo / 'results/clinical_evaluation')
    HERE = str(_repo / 'analyses/clinical_evaluation')
    ALPHA = GOLD
    RESEARCH = GOLD
    TABLES = path_join(PAPER, 'tables')
    FIGURES = path_join(PAPER, 'figures')
    os.makedirs(TABLES, exist_ok=True)
    os.makedirs(FIGURES, exist_ok=True)
    DATASET = str(_repo / 'data' / 'limfmed_dataset_gold.csv')
    PBASE = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    PBASE_NON = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_truncated'
    ORG_DS = 'experiments/clinical_protocols/delphi_panels/heterogeneous_deepseek_organizer_complete'
    ORG_QWEN = 'experiments/clinical_protocols/delphi_panels/heterogeneous_qwen_organizer_complete'
    ORG_GEM = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gemini_external_organizer'
    PWEAK = 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro'
    PMIXED = 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm'
    PSTRONG = 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'
    HOMOG = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'deepseekv4': 'experiments/homogeneous_panels/three_deepseekv4_members_complete', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    FIX_PBASE = 'experiments/fixed_length_controls/delphi_panels/internal_panel_variants/standard_protocol/heterogeneous_gpt120_organizer'
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    MODEL_NAME = {'gpt': 'gpt-oss-120b', 'qwen': 'qwen3-235b-a22b', 'deepseekv4': 'deepseek-v4-flash', 'deepseekv4pro': 'deepseek-v4-pro', 'gemini': 'gemini-3.7-flash', 'glm': 'glm-5.2', 'kimi': 'kimi'}
    PANELS = {'P-base': (PBASE, 'delphi_base_{d}_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'gpt', True), 'P-base/org-ds': (ORG_DS, 'delphi_base_{d}_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'deepseekv4', True), 'P-base/org-qwen': (ORG_QWEN, 'delphi_base_{d}_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'qwen', True), 'P-base/org-gemini': (ORG_GEM, 'delphi_base_{d}_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'gemini', False), 'P-weak': (PWEAK, 'delphi_pweak_{d}_judge.csv', ['qwen', 'gemini', 'deepseekv4pro'], 'qwen', True), 'P-mixed': (PMIXED, 'delphi_pmixed_{d}_judge.csv', ['deepseekv4', 'gemini', 'glm'], 'deepseekv4', True), 'P-strong': (PSTRONG, 'delphi_pstrong_{d}_judge.csv', ['deepseekv4', 'glm', 'gpt'], 'deepseekv4', True), 'homog gpt': (HOMOG['gpt'], 'delphi_homog_gpt_{d}_judge.csv', ['gpt'] * 3, 'gpt', True), 'homog qwen': (HOMOG['qwen'], 'delphi_homog_qwen_{d}_judge.csv', ['qwen'] * 3, 'qwen', True), 'homog deepseekv4': (HOMOG['deepseekv4'], 'delphi_homog_deepseekv4_{d}_judge.csv', ['deepseekv4'] * 3, 'deepseekv4', True), 'homog gemini': (HOMOG['gemini'], 'delphi_homog_gemini_{d}_judge.csv', ['gemini'] * 3, 'gemini', True)}
    HETERO = ['P-base', 'P-weak', 'P-mixed', 'P-strong']
    HOMOGS = ['homog gpt', 'homog qwen', 'homog deepseekv4', 'homog gemini']
    ORGANIZER_ARMS = ['P-base', 'P-base/org-ds', 'P-base/org-qwen', 'P-base/org-gemini']
    RAW_RUN = {'P-base': PBASE + '/delphi_noroles.csv', 'P-base/org-ds': ORG_DS + '/delphi_noroles.csv', 'P-base/org-qwen': ORG_QWEN + '/delphi_noroles.csv', 'P-base/org-gemini': ORG_GEM + '/delphi_noroles.csv', 'P-weak': PWEAK + '/delphi_pweak_noroles.csv', 'P-mixed': PMIXED + '/delphi_pmixed_noroles.csv', 'P-strong': PSTRONG + '/delphi_pstrong_noroles.csv', 'homog gpt': HOMOG['gpt'] + '/delphi_homog_gpt_noroles.csv', 'homog qwen': HOMOG['qwen'] + '/delphi_homog_qwen_noroles.csv', 'homog deepseekv4': HOMOG['deepseekv4'] + '/delphi_homog_deepseekv4_noroles.csv', 'homog gemini': HOMOG['gemini'] + '/delphi_homog_gemini_noroles.csv'}
    ROUNDS_EVAL = {'P-base': (PBASE, 'delphi_base_noroles_protocol_r{r}_judge.csv'), 'P-weak': (PWEAK, 'delphi_pweak_noroles_protocol_r{r}_judge.csv'), 'P-mixed': (PMIXED, 'delphi_pmixed_noroles_protocol_r{r}_judge.csv'), 'P-strong': (PSTRONG, 'delphi_pstrong_noroles_protocol_r{r}_judge.csv'), 'homog gpt': (HOMOG['gpt'], 'delphi_homog_gpt_noroles_protocol_r{r}_judge.csv'), 'homog gemini': (HOMOG['gemini'], 'delphi_homog_gemini_noroles_protocol_r{r}_judge.csv'), 'homog deepseekv4': (HOMOG['deepseekv4'], 'delphi_homog_deepseekv4_noroles_protocol_r{r}_judge.csv')}

    def baseline_path(model, design='noroles', variant='nofix'):
        return f"experiments/{('clinical_protocols' if variant == 'nofix' else 'fixed_length_controls')}/single_model_baselines/baseline_{model}/llm_as_judge/baseline_{design}_{model}_judge.csv"

    def panel_path(name, design='noroles'):
        d, f, *_ = PANELS[name]
        return d + '/llm_as_judge/' + f.format(d=design)

    def _abs(rel):
        return rel if os.path.isabs(rel) else path_join(GOLD, rel)

    def read_csv(rel, filter_result=True):
        p = _abs(rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        d.columns = [c.lstrip('\ufeff') for c in d.columns]
        if filter_result and 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        return d

    def read_judge(rel):
        d = read_csv(rel)
        if d is None or 'Judge_score' not in d.columns:
            return None
        d = d.dropna(subset=['Judge_score', 'Vignette']).drop_duplicates('Vignette', keep='first')
        d['Judge_score'] = pd.to_numeric(d['Judge_score'], errors='coerce')
        return d.set_index('Vignette')

    def load_main_arms(design='noroles', include_usc=False):
        arms = {}
        for m in SINGLES:
            d = read_judge(baseline_path(m, design))
            if d is not None:
                arms[m] = d
        for name in PANELS:
            d = read_judge(panel_path(name, design))
            if d is not None:
                arms[name] = d
        if include_usc and design == 'noroles':
            for k in (3, 5, 7):
                d = read_judge(f'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc{k}_judge.csv')
                if d is not None:
                    arms[f'USC@{k} gpt'] = d
        return arms

    def score(arms, name):
        return arms[name]['Judge_score'].astype(float)

    def dataset():
        d = normalize_newlines(pd.read_csv(DATASET, sep=None, engine='python'))
        return d

    def protocol_text(d):
        for c in ('winner_protocol', 'baseline_protocol_anonymous', 'baseline_protocol'):
            if c in d.columns:
                return d[c].astype(str)
        raise KeyError('no protocol column')

    def strip_author(text):
        if not isinstance(text, str):
            return ''
        return re.split('(?im)^\\s*Protocol Author\\s*:', text, maxsplit=1)[0].rstrip()

    def n_words(text):
        return len(re.findall('\\S+', strip_author(text))) if isinstance(text, str) else np.nan

    def items(d):
        rows = []
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                rows.append((v, int(it['id']), int(it.get('points', 0) or 0), ax, bool(it.get('met'))))
        return pd.DataFrame(rows, columns=['Vignette', 'cid', 'points', 'axis', 'met'])

    def item_matrix(arms, names, positive_only=True):
        frames = []
        for n in names:
            it = items(arms[n]).set_index(['Vignette', 'cid'])
            frames.append(it['met'].astype(int).rename(n))
            meta = it[['points', 'axis']]
        m = pd.concat(frames, axis=1, join='inner')
        m = m.join(meta, how='left')
        if positive_only:
            m = m[m['points'] > 0]
        return m
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        x = x.strip()
        if x in GT:
            return GT[x]
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in (('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')):
            if k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        return norm_label(m.group(1)) if m else None

    def predicted_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        return protocol_text(d).map(parse_triage)

    def true_triage(d):
        return d['Type'].map(GT)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def paired(a, b):
        ix = a.dropna().index.intersection(b.dropna().index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        dd = x - y
        p = stats.wilcoxon(x, y).pvalue if (dd != 0).any() else np.nan
        return {'n': len(ix), 'delta': float(dd.mean()), 'share_gt': float((dd > 0).mean()), 'p_wilcoxon': float(p)}

    def boot_ci(x, B=10000, seed=20260922, alpha=0.05, stat=np.mean):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        idx = rng.integers(0, n, (B, n))
        s = stat(x[idx], axis=1)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_two_level(x, cluster, B=10000, seed=20260922, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c')]
        G = len(groups)
        out = np.empty(B)
        for b in range(B):
            pick = rng.integers(0, G, G)
            vals = []
            for i in pick:
                g = groups[i]
                vals.append(g[rng.integers(0, len(g), len(g))])
            out[b] = np.concatenate(vals).mean()
        return (float(np.quantile(out, alpha / 2)), float(np.quantile(out, 1 - alpha / 2)))

    def holm(p):
        p = np.asarray(p, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def mcnemar_exact(a, b):
        ix = a.dropna().index.intersection(b.dropna().index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n10, n01 = (int((x & ~y).sum()), int((~x & y).sum()))
        p = stats.binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
        return {'n': len(ix), 'acc_a': float(x.mean()), 'acc_b': float(y.mean()), 'n10': n10, 'n01': n01, 'p': float(p)}

    def wilson(k, n, z=1.96):
        p = k / n
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def tost_paired(dvec, margin):
        d = np.asarray(dvec, float)
        n = len(d)
        m, se = (d.mean(), d.std(ddof=1) / np.sqrt(n))
        p_low = stats.t.sf((m + margin) / se, n - 1)
        p_up = stats.t.cdf((m - margin) / se, n - 1)
        return float(max(p_low, p_up))

    def fmt_p(p):
        if p is None or not np.isfinite(p):
            return 'n/a'
        if p < 0.0001:
            return f'{p:.1e}'
        return f'{p:.3f}' if p >= 0.001 else f'{p:.4f}'

    def save_table(df, name, index=True):
        path = path_join(TABLES, name)
        df.to_csv(path, sep=';', encoding='utf-8-sig', index=index)
        print(f'saved tables/{name} ({len(df)} rows)')
        return path
    return locals()
from types import SimpleNamespace
P = SimpleNamespace(**_build_data_tools())
del _build_data_tools
pd.set_option('display.width', 220)
arms = P.load_main_arms()


In [2]:
for n in ['P-base', 'P-strong', 'homog gpt']:
    d = arms[n]
    txt = P.protocol_text(d).map(P.parse_triage)
    con = d.delphi_consensus_emergency_type.map(P.norm_label)
    print(n, 'consensus label == label written in the protocol:', round(float((txt == con).mean()), 4))


P-base consensus label == label written in the protocol: 1.0
P-strong consensus label == label written in the protocol: 1.0


homog gpt consensus label == label written in the protocol: 1.0


In [3]:
rows = []
for n, d in arms.items():
    t, p = (P.true_triage(d), P.predicted_triage(d))
    ok = p.notna() & t.notna()
    o = p[ok].map(P.ORD) - t[ok].map(P.ORD)
    em = t[ok] == 'EMERG'
    rows.append({'arm': P.MODEL_NAME.get(n, n), 'key': n, 'n': int(ok.sum()), 'unparsed': int((~p.notna()).sum()), 'accuracy': float((o == 0).mean()), 'under_triage': float((o < 0).mean()), 'over_triage': float((o > 0).mean()), 'emergent_missed': float((o[em] < 0).mean()), 'two_step_error': float((o.abs() == 2).mean())})
tri = pd.DataFrame(rows).sort_values('accuracy', ascending=False)
display(tri.round(4))
P.save_table(tri, 'triage_accuracy_and_safety.csv', index=False)


,arm,key,n,unparsed,accuracy,under_triage,over_triage,emergent_missed,two_step_error
1,qwen3-235b-a22b,qwen,229,0,0.8952,0.0218,0.0830,0.0119,0.0000
2,deepseek-v4-flash,deepseekv4,228,0,0.8860,0.0219,0.0921,0.0238,0.0000
3,deepseek-v4-pro,deepseekv4pro,228,1,0.8816,0.0175,0.1009,0.0238,0.0000
4,gemini-3.7-flash,gemini,229,0,0.8603,0.0131,0.1266,0.0238,0.0000
5,glm-5.2,glm,228,1,0.8553,0.0088,0.1360,0.0238,0.0000
12,P-mixed,P-mixed,229,0,0.8384,0.0087,0.1528,0.0238,0.0000
9,P-base/org-qwen,P-base/org-qwen,229,0,0.8297,0.0044,0.1659,0.0119,0.0000
16,homog deepseekv4,homog deepseekv4,229,0,0.8253,0.0131,0.1616,0.0357,0.0000
13,P-strong,P-strong,229,0,0.8253,0.0131,0.1616,0.0238,0.0000
8,P-base/org-ds,P-base/org-ds,229,0,0.8253,0.0087,0.1659,0.0238,0.0000


saved tables/triage_accuracy_and_safety.csv (18 rows)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\results\\clinical_evaluation\\tables\\triage_accuracy_and_safety.csv'

In [4]:
def correct(n):
    d = arms[n]
    return P.predicted_triage(d) == P.true_triage(d)
rows = []
for a in ['P-base', 'P-mixed', 'P-strong', 'P-base/org-ds', 'P-base/org-qwen', 'P-base/org-gemini']:
    for b in ['qwen', 'deepseekv4', 'gpt']:
        m = P.mcnemar_exact(correct(a), correct(b))
        rows.append({'panel': a, 'single': P.MODEL_NAME[b], **m})
mc = pd.DataFrame(rows)
mc['p_holm'] = P.holm(mc.p)
display(mc.round(4))
P.save_table(mc, 'paired_triage_accuracy_tests.csv', index=False)


,panel,single,n,acc_a,acc_b,n10,n01,p,p_holm
0,P-base,qwen3-235b-a22b,229,0.7860,0.8952,13,38,0.0006,0.0106
1,P-base,deepseek-v4-flash,228,0.7851,0.8860,6,29,0.0001,0.0021
2,P-base,gpt-oss-120b,229,0.7860,0.8166,13,20,0.2962,1.0000
3,P-mixed,qwen3-235b-a22b,229,0.8384,0.8952,14,27,0.0596,0.4179
4,P-mixed,deepseek-v4-flash,228,0.8377,0.8860,8,19,0.0522,0.4179
5,P-mixed,gpt-oss-120b,229,0.8384,0.8166,17,12,0.4583,1.0000
6,P-strong,qwen3-235b-a22b,229,0.8253,0.8952,13,29,0.0195,0.2147
7,P-strong,deepseek-v4-flash,228,0.8246,0.8860,6,20,0.0094,0.1403
8,P-strong,gpt-oss-120b,229,0.8253,0.8166,14,12,0.8450,1.0000
9,P-base/org-ds,qwen3-235b-a22b,229,0.8253,0.8952,15,31,0.0259,0.2370


saved tables/paired_triage_accuracy_tests.csv (18 rows)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\results\\clinical_evaluation\\tables\\paired_triage_accuracy_tests.csv'

In [5]:
T = tri.set_index('key')
rows = []
for a in ['P-base', 'P-mixed', 'P-strong', 'P-base/org-ds', 'P-base/org-qwen', 'P-base/org-gemini']:
    for b in ['qwen', 'deepseekv4', 'gpt']:
        du = T.loc[b, 'under_triage'] - T.loc[a, 'under_triage']
        do = T.loc[a, 'over_triage'] - T.loc[b, 'over_triage']
        R = do / du if du > 0 else np.inf if do > 0 else 0
        rows.append({'panel': a, 'vs': P.MODEL_NAME[b], 'under_panel': T.loc[a, 'under_triage'], 'under_single': T.loc[b, 'under_triage'], 'over_panel': T.loc[a, 'over_triage'], 'over_single': T.loc[b, 'over_triage'], 'indifference_R': R})
ind = pd.DataFrame(rows)
display(ind.round(3))
P.save_table(ind, 'triage_cost_indifference_thresholds.csv', index=False)


,panel,vs,under_panel,under_single,over_panel,over_single,indifference_R
0,P-base,qwen3-235b-a22b,0.013,0.022,0.201,0.083,13.500
1,P-base,deepseek-v4-flash,0.013,0.022,0.201,0.092,12.319
2,P-base,gpt-oss-120b,0.013,0.048,0.201,0.135,1.875
3,P-mixed,qwen3-235b-a22b,0.009,0.022,0.153,0.083,5.333
4,P-mixed,deepseek-v4-flash,0.009,0.022,0.153,0.092,4.602
5,P-mixed,gpt-oss-120b,0.009,0.048,0.153,0.135,0.444
6,P-strong,qwen3-235b-a22b,0.013,0.022,0.162,0.083,9.000
7,P-strong,deepseek-v4-flash,0.013,0.022,0.162,0.092,7.868
8,P-strong,gpt-oss-120b,0.013,0.048,0.162,0.135,0.750
9,P-base/org-ds,qwen3-235b-a22b,0.009,0.022,0.166,0.083,6.333


saved tables/triage_cost_indifference_thresholds.csv (18 rows)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\results\\clinical_evaluation\\tables\\triage_cost_indifference_thresholds.csv'

In [6]:
import importlib
d = P.read_csv(P.RAW_RUN['P-base']).dropna(subset=['Vignette']).drop_duplicates('Vignette').set_index('Vignette')
pids = P.participants(d)
from collections import Counter

def plur(r):
    lab = [P.norm_label(r.get(f'r1_emergency__{p}')) for p in pids]
    lab = [x for x in lab if x]
    if not lab:
        return None
    c = Counter(lab)
    top = max(c.values())
    tied = [k for k, v in c.items() if v == top]
    if len(tied) == 1:
        return tied[0]
    o = P.norm_label(r.get('r1_emergency__gpt120b_default'))
    return o if o in tied else tied[0]
m1 = d.apply(plur, axis=1)
t = d.Type.map(P.GT)
fin = d.delphi_consensus_emergency_type.map(P.norm_label)
for lab, s in [('R1 majority of members', m1), ('final consensus', fin)]:
    o = s.map(P.ORD) - t.map(P.ORD)
    print(f'{lab:<24} acc {(o == 0).mean():.3f} under {(o < 0).mean():.3f} over {(o > 0).mean():.3f}')
for p in pids:
    o = d[f'r1_emergency__{p}'].map(P.norm_label).map(P.ORD) - t.map(P.ORD)
    print(f'   R1 {p:<20} acc {(o == 0).mean():.3f} under {(o < 0).mean():.3f} over {(o > 0).mean():.3f}')


R1 majority of members   acc 0.821 under 0.004 over 0.175
final consensus          acc 0.786 under 0.013 over 0.201
   R1 deepseekv4_default   acc 0.843 under 0.009 over 0.148
   R1 gpt120b_default      acc 0.721 under 0.044 over 0.236
   R1 qwen_default         acc 0.755 under 0.000 over 0.236
